In [1]:
import json
import faiss
import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

h:\Rag Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 1. Dataset load karein (Hugging Face se)
print("Dataset load ho raha hai...")
ds = load_dataset("semihk1/aws-public-pdf-chunked-dataset", split="train")

Dataset load ho raha hai...


Generating train split: 100%|██████████| 106621/106621 [00:07<00:00, 14208.73 examples/s]


In [4]:
# Sirf pehle 5000 chunks lein, taaki 2 din ke andar test ho sake, poora data bahut bada hai
# Jab deploy karein tab adjust kar sakte hain
texts = [item["text"] for item in ds.select(range(min(5000, len(ds))))]
sources = [item["source"] for item in ds.select(range(min(5000, len(ds))))]

print(f"Total {len(texts)} chunks load hue")

Total 5000 chunks load hue


In [5]:
# 2. Embeddings model load karein (local, free)
print("Embedding model load ho raha hai...")
model = SentenceTransformer("all-MiniLM-L6-v2")

Embedding model load ho raha hai...


h:\Rag Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abhi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3714.81it/s]


In [6]:
# 3. Embeddings generate karein
print("Embeddings generate ho rahe hain...")
embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)

Embeddings generate ho rahe hain...


Batches: 100%|██████████| 79/79 [02:00<00:00,  1.53s/it]


In [7]:
# 4. FAISS index banayein aur save karein
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))

faiss.write_index(index, "aws_index.faiss")

In [8]:
# 5. Metadata save karein (retrieval ke waqt citation ke liye use hoga)
with open("metadata.json", "w") as f:
    json.dump({"texts": texts, "sources": sources}, f)

print("Ho gaya! aws_index.faiss aur metadata.json save ho gaye")

Ho gaya! aws_index.faiss aur metadata.json save ho gaye
